# Новый эксперимент: готовая таблица → Python-преобразования Llama → LightGBM

# LLM-FE на H&M: Llama‑3.1‑8B‑Instruct, Kaggle T4

Llama преобразует готовую плоскую таблицу30 столбцов одной Python-функцией; LightGBM измеряет их качество и направляет
поиск в трёх популяциях. LightGBM используется на всех этапах: контроль, search,
confirmation и итоговая оценка. Ручной LightGBM и GNN→LightGBM сравниваются по сохранённым предсказаниям
на **тех же пользователях и кандидатах**. Это адаптация исследования, не его точное воспроизведение.

**Как запустить:**
1. В Kaggle импортируйте этот `.ipynb` через **File → Import Notebook**.
2. Выберите GPU **T4** (достаточно одной; при T4×2 используется GPU0), включите Internet.
3. Создайте Kaggle Dataset из `hm_llmfe_kaggle_fulltrain_v2.zip` и добавьте его через Input.
   Поддерживаются и исходный ZIP, и автоматически распакованные Kaggle файлы.
4. В конфиге выберите `MODEL_SOURCE = 'hf'` (по умолчанию, без токена) или `'hf'`.
   Для `'hf'` получите доступ к [официальной Llama](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct)
   и добавьте `HF_TOKEN` в **Add-ons → Secrets**. Оба источника размещены на Hugging Face.
5. Выполните **Run All**. Модель скачивается автоматически; веса не входят в архив данных.
6. Скачайте итоговый `*_results.zip` из Output. В нём программы, модели деревьев, метрики и графики.

В ZIP данных находятся **те же803377 обучающих пар /50000 user-time строк /52 снимка**, что
в локальных сравнениях. Новое отрицательное семплирование не выполняется.
Ограничена только оценка: search256, confirmation512, отдельная val1024, test2048 пользователей.
Поисковые/confirmation/итоговые val строки не пересекаются. Кандидаты ID-GNN неизменны;
позитивные test-товары в пул дополнительно не добавляются.

Контроли: Raw15, Prepared30 (15raw+15 ручных исторических агрегатов), Seeded32 (ещё2 ручных log-признака). LLM-поиск сравнивается с этими контролями; начальный пример уже обучен и измерен. Первые6 обращений — пилот: при0 успешных наборов остановка с ZIP; иначе поиск20/300/3 продолжается. Подготовленная таблица кэшируется в working; исходный Input-архив прежний.


## Условия и отличия от статьи

| Параметр | Этот ноутбук |
|---|---|
| LLM | Llama-3.1-8B-Instruct: официальный источник Meta (`hf`) или публикация Unsloth (`unsloth`) |
| Размещение на T4 | NF4 4-bit, вычисления float16; квантизация — наше отличие |
| Генерация | temperature0.8,3 новых предложения на один prompt; цель20 различных оценённых наборов |
| Бюджет/исправления | До300 обращений суммарно, включая до3 исправлений одного предложения |
| Память поиска | 3 независимые популяции; до3 демонстраций, кластеры по validation MAP, Boltzmann sampling |
| Форма признаков | modify_features(X): готовая таблица, без history/SQL/joins; 15 фиксированных исторических агрегатов host; без GNN |
| До обучения | Проверка на малых раннем/позднем train-снимках и search-строках; не считается оценённым набором |
| Основной evaluator | LightGBM, binary objective,150 boosting rounds,31 лист; подбор через MAP@12 |
| Проверка сопоставимости | Raw/LLM LightGBM на одних парах; те же деревья проходят confirmation и финал |
| История | Только выбранная когорта клиентов, до cutoff, последние180 дней; global counts относятся к этой когорте |
| Ресурсы | 30секунд на вызов функции признаков,4GiB на отдельный evaluator,1800секунд на обучение+оценку |
| Отбор | search→confirmation; итоговые val/test открываются после фиксации программы |

В статье рассматриваются табличные classification/regression, используются другие датасеты,
а лимит30секунд относится к полной оценке. Здесь сохраняем оригинальные803377 пар, поэтому
выделяем отдельный больший лимит для полного обучения и4GiB вместо2GiB. По прямому указанию
пользователя весь отбор переведён на LightGBM; XGBoost в этом протоколе не используется.
В Kaggle нет повторного обучения GNN. Это новый протокол flat v1 с проверкой интерфейса, бюджетом успешных наборов
и исправлениями, отличающийся от20 сэмплов статьи. Старые результаты не переиспользуются.
Референсные деревья использовали те же пары, но другие схемы подбора/производные seeds;
GNN дополнительно обучалась на графе. Это сравнение качества конвейеров, не изолированный
эффект способа генерации признаков. Полные старые MAP нельзя подставлять вместо MAP этой выборки.

Источники: [LLM-FE §4.3, Appendix B](https://arxiv.org/html/2503.14434v3),
[код авторов](https://github.com/nikhilsab/LLMFE),
[NF4](https://huggingface.co/docs/transformers/v4.57.1/quantization/bitsandbytes).


In [ ]:
import subprocess, sys
# Не переустанавливаем torch/CUDA из готового окружения Kaggle.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'accelerate==1.10.1', 'bitsandbytes==0.48.1',
    'lightgbm==4.6.0', 'sentencepiece==0.2.1','duckdb==1.4.1'])


In [ ]:
from pathlib import Path
import os, json, hashlib, time, gc, importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image, FileLink

# Оба варианта скачиваются с Hugging Face; различается издатель репозитория.
MODEL_SOURCE = 'hf'  # 'unsloth' без токена; 'hf' официальный Meta с доступом/HF_TOKEN
MODEL_SOURCES = {
    'hf': {'repo_id':'meta-llama/Llama-3.1-8B-Instruct', 'requires_token':True},
    'unsloth': {'repo_id':'unsloth/Meta-Llama-3.1-8B-Instruct', 'requires_token':False},
}
if MODEL_SOURCE not in MODEL_SOURCES:
    raise ValueError("MODEL_SOURCE должен быть 'hf' или 'unsloth'")
MODEL_ID = MODEL_SOURCES[MODEL_SOURCE]['repo_id']

def resolve_auth(source, environ, load_secret):
    if source not in MODEL_SOURCES:
        raise ValueError("MODEL_SOURCE должен быть 'hf' или 'unsloth'")
    if not MODEL_SOURCES[source]['requires_token']:
        # False запрещает также автоматическое использование старого токена из HF-кэша.
        return False
    value = environ.get('HF_TOKEN')
    if not value:
        try:
            value = load_secret()
        except Exception:
            value = None
    if not value:
        raise RuntimeError('Для hf нужны одобренный доступ Meta и Kaggle Secret HF_TOKEN; либо выберите unsloth')
    return value

# При повторе можно указать SHA из предыдущего run_config.json; пустая строка разрешит текущую ревизию.
MODEL_REVISION = '0e9e39f249a16976918f6564b8830bc894c89659' if MODEL_SOURCE=='hf' else ''
WORK = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(WORK / 'hf_cache')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

CONFIG = {
    'protocol': 'hm_llmfe_llama31_8b_lightgbm_flat_fulltrain_v1', 'seed':42,
    'search_backend':'lightgbm',
    'model_id':MODEL_ID, 'model_source':MODEL_SOURCE,
    'quantization':'bnb NF4 4bit/double_quant, FP16 compute',
    'temperature':0.8, 'top_p':1.0, 'top_k':0,
    'islands':3, 'programs_per_prompt':3, 'max_demonstrations':3,
    'target_successful_distinct':20, 'max_llm_calls':300, 'max_repairs_per_proposal':3,
    'program_interface_version':1,'feature_interface':'flat_python','abort_if_no_success_after_calls':6, 'preflight_before_full_training':True,
    'max_input_tokens':6144, 'max_new_tokens':2048,
    'train_pairs':803377, 'train_user_times':50000,
    'search_users':256, 'confirmation_users':512, 'heldout_val_users':1024, 'test_users':2048,
    'cpu_threads':min(4, os.cpu_count() or 1),
    'evaluation_memory_gb':4, 'evaluation_timeout_seconds':1800,
    'feature_call_timeout_seconds':30, 'lightgbm_rounds':150,
    'paper_reproduction':False, 'candidate_protocol':'exact saved ID-GNN seed42 pools',
}


In [ ]:
# Код evaluator встроен в ноутбук: клонировать репозиторий не требуется.
MODULES = {'worker.py': '"""CPU-only bounded evaluator for LLM-authored pandas feature programs."""\nimport os\nif __name__==\'__main__\':\n    os.environ[\'CUDA_VISIBLE_DEVICES\']=\'\'\n    for key in [\'OMP_NUM_THREADS\',\'MKL_NUM_THREADS\',\'OPENBLAS_NUM_THREADS\',\'NUMEXPR_NUM_THREADS\']:\n        os.environ[key]=\'1\'\nimport argparse\nimport ast\nimport hashlib\nimport gc\nimport json\nfrom pathlib import Path\nimport resource\nimport signal\nimport time\nimport traceback\nimport numpy as np\nimport pandas as pd\n\nfrom flat_features import RAW, SEED, RAW_ONLY, Data, matrix, compile_program, check_program\nDISALLOWED_CALLS={\'open\',\'eval\',\'exec\',\'compile\',\'getattr\',\'setattr\',\'delattr\',\'globals\',\'locals\',\'vars\',\n                  \'input\',\'breakpoint\',\'help\',\'dir\',\'memoryview\',\'type\',\'super\',\'__import__\'}\nDISALLOWED_ATTRIBUTES={\'eval\',\'query\',\'read_csv\',\'read_parquet\',\'read_pickle\',\'read_json\',\'read_sql\',\n                        \'read_excel\',\'read_html\',\'read_feather\',\'load\',\'loads\',\'save\',\'savetxt\',\'savez\',\n                        \'savez_compressed\',\'fromfile\',\'tofile\',\'loadtxt\',\'genfromtxt\',\'memmap\',\'ctypes\',\n                        \'f2py\',\'testing\',\'lib\',\'io\',\'compat\'}\n\n\n\n\n\n\ndef feature_error(error,func,X,history):\n    """Actionable model feedback without a long pandas traceback or any labels."""\n    frames=[f for f in traceback.extract_tb(error.__traceback__) if f.filename==\'<llm-features>\']\n    lines=getattr(func,\'_feature_source\',\'\').splitlines()\n    location=\'\'\n    if frames:\n        number=frames[-1].lineno\n        location=\'\\nYour code near line \'+str(number)+\':\\n\'+\'\\n\'.join(\n            f\'{i+1}: {lines[i]}\' for i in range(max(0,number-3),min(len(lines),number+2)))\n    return (f\'{type(error).__name__}: {error}\'+location+\n            \'\\nAvailable INPUT X columns: \'+str(list(X.columns))+\n            \'\\nAvailable INPUT history columns: \'+str(list(history.columns))+\n            \'\\nIntermediate tables may have different columns: inspect your groupby/agg/reset_index/merge. \'\n            \'Overlapping non-key columns get _x/_y suffixes after merge. \'\n            \'History has no raw_u_*/raw_i_* columns. Return only f_* features in unchanged X order.\')\n\n\n\n\n\n\ndef map_at_12(predictions,targets):\n    if len(predictions)!=len(targets):\n        raise ValueError(\'User alignment changed\')\n    hits=np.array([np.isin(p,t) for p,t in zip(predictions,targets)])\n    denominator=np.minimum(12,np.array([len(np.unique(t)) for t in targets]))\n    if (denominator==0).any():\n        raise ValueError(\'Empty target list\')\n    ap=((hits.cumsum(axis=1)/np.arange(1,13))*hits).sum(axis=1)/denominator\n    return float(ap.mean()),ap\n\n\ndef rank(scores,pairs,targets):\n    predictions=[]\n    if list(pairs.customer_id.drop_duplicates())!=list(targets.customer_id):\n        raise ValueError(\'Pool/user order changed\')\n    for _,frame in pairs.assign(score=scores).groupby(\'customer_id\',sort=False):\n        if len(frame)<12 or frame.article_id.duplicated().any():\n            raise ValueError(\'Invalid candidate pool\')\n        predictions.append(frame.sort_values([\'score\',\'article_id\'],ascending=[False,True]).article_id.head(12).to_numpy())\n    return np.stack(predictions)\n\n\ndef limits(memory_gb):\n    size=int(memory_gb*1024**3)\n    resource.setrlimit(resource.RLIMIT_AS,(size,size))\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--data\',required=True)\n    parser.add_argument(\'--directory\',required=True)\n    parser.add_argument(\'--stage\',choices=[\'preflight\',\'search\',\'confirm\',\'val\',\'test\',\'replay_val\',\'replay_test\'],required=True)\n    parser.add_argument(\'--threads\',type=int,default=4)\n    parser.add_argument(\'--memory-gb\',type=float,default=4)\n    parser.add_argument(\'--backend\',choices=[\'xgboost\',\'lightgbm\'],default=\'lightgbm\')\n    args=parser.parse_args()\n    limits(args.memory_gb)\n    tick=time.perf_counter()\n    destination=Path(args.directory)\n    source=(destination/\'program.py\').read_text()\n    func=compile_program(source)\n    data=Data(args.data)\n    if args.stage==\'preflight\':\n        train=pd.read_parquet(data.root/\'train_pairs.parquet\')\n        names=None\n        checked=[]\n        for timestamp in [train.timestamp.min(),train.timestamp.max()]:\n            sample=train[train.timestamp==timestamp].head(256).copy().reset_index(drop=True)\n            values,names=matrix(data,sample,func,expected=names)\n            middle=len(sample)//2\n            halves=np.concatenate([matrix(data,sample.iloc[:middle],func,expected=names)[0],matrix(data,sample.iloc[middle:],func,expected=names)[0]])\n            if not np.array_equal(values,halves,equal_nan=True):raise ValueError(\'Features depend on batch size or other rows; use rowwise transformations only\')\n            checked.append({\'kind\':\'train\',\'cutoff\':str(timestamp),\'pairs\':len(sample)})\n        sample=pd.read_parquet(data.root/\'search_pairs.parquet\').head(256).reset_index(drop=True)\n        values,names=matrix(data,sample,func,expected=names)\n        middle=len(sample)//2\n        halves=np.concatenate([matrix(data,sample.iloc[:middle],func,expected=names)[0],matrix(data,sample.iloc[middle:],func,expected=names)[0]])\n        if not np.array_equal(values,halves,equal_nan=True):raise ValueError(\'Features depend on batch size or other rows; use rowwise transformations only\')\n        checked.append({\'kind\':\'search_validation\',\'pairs\':len(sample)})\n        if not getattr(func,\'_feature_changed\',False):\n            raise ValueError(\'Program generated no new features\')\n        result={\'stage\':\'preflight\',\'passed\':True,\'smoke\':True,\'backend\':args.backend,\n                \'program_sha256\':hashlib.sha256(source.encode()).hexdigest(),\n                \'checked\':checked,\'features\':names,\'seconds\':time.perf_counter()-tick,\n                \'counted_as_successful_set\':False}\n        (destination/\'preflight_metrics.json\').write_text(json.dumps(result,indent=2))\n        print(json.dumps(result))\n        return\n    model_path=destination/(\'model.json\' if args.backend==\'xgboost\' else \'model.txt\')\n    if args.stage==\'search\' and not model_path.exists():\n        pairs=pd.read_parquet(data.root/\'train_pairs.parquet\')\n        values,names=matrix(data,pairs,func)\n        if source!=RAW and not getattr(func,\'_feature_changed\',False):\n            raise ValueError(\'Program generated no new features\')\n        categorical=set(data.raw_i+[\'raw_u_club_member_status\',\'raw_u_fashion_news_frequency\'])\n        kinds=[\'c\' if n in categorical else \'q\' for n in names]\n        for i,kind in enumerate(kinds):\n            if kind==\'c\':\n                values[values[:,i]<0,i]=np.nan\n        history={}\n        if args.backend==\'xgboost\':\n            import xgboost as xgb\n            dataset=xgb.QuantileDMatrix(values,label=pairs.label.values,feature_names=names,feature_types=kinds,\n                                       enable_categorical=True,nthread=args.threads,max_bin=127)\n            params={\'objective\':\'binary:logistic\',\'eval_metric\':\'logloss\',\'tree_method\':\'hist\',\'device\':\'cpu\',\n                \'max_depth\':6,\'eta\':.05,\'subsample\':.8,\'colsample_bytree\':.9,\'max_bin\':127,\n                \'seed\':42,\'nthread\':args.threads,\'max_cached_hist_node\':64}\n            model=xgb.train(params,dataset,num_boost_round=150,evals=[(dataset,\'train\')],evals_result=history,verbose_eval=False)\n            model.save_model(model_path)\n        else:\n            import lightgbm as lgb\n            params={\'objective\':\'binary\',\'metric\':\'binary_logloss\',\'verbosity\':-1,\'num_threads\':args.threads,\n                    \'learning_rate\':.05,\'num_leaves\':31,\'min_data_in_leaf\':100,\'max_bin\':127,\n                    \'feature_fraction\':.9,\'bagging_fraction\':.8,\'bagging_freq\':1,\'deterministic\':True,\n                    \'force_col_wise\':True,\'seed\':42,\'feature_fraction_seed\':42,\'bagging_seed\':42,\'data_random_seed\':42}\n            dataset=lgb.Dataset(values,label=pairs.label.values,feature_name=names,categorical_feature=[n for n in names if n in categorical])\n            model=lgb.train(params,dataset,num_boost_round=150,valid_sets=[dataset],valid_names=[\'train\'],callbacks=[lgb.record_evaluation(history)])\n            history={\'train\':{\'logloss\':history[\'train\'][\'binary_logloss\']}}\n            model.save_model(str(model_path))\n        (destination/\'schema.json\').write_text(json.dumps({\'features\':names,\'types\':kinds,\'params\':params,\'train_pairs\':len(pairs),\n                                                          \'item_category_maps\':data.category_maps}))\n        (destination/\'learning_curve.json\').write_text(json.dumps(history))\n        del pairs,values,dataset,model\n        gc.collect()\n    if args.backend==\'xgboost\':\n        import xgboost as xgb\n        model=xgb.Booster()\n        model.load_model(model_path)\n        model.set_param({\'nthread\':args.threads,\'device\':\'cpu\'})\n    else:\n        import lightgbm as lgb\n        model=lgb.Booster(model_file=str(model_path))\n    schema=json.loads((destination/\'schema.json\').read_text())\n    split=args.stage.replace(\'replay_\',\'\')\n    pairs=pd.read_parquet(data.root/f\'{split}_pairs.parquet\')\n    targets=pd.read_parquet(data.root/f\'{split}_targets.parquet\')\n    values,names=matrix(data,pairs,func,expected=schema[\'features\'])\n    for i,kind in enumerate(schema[\'types\']):\n        if kind==\'c\':\n            values[values[:,i]<0,i]=np.nan\n    if args.backend==\'xgboost\':\n        scores=model.predict(xgb.DMatrix(values,feature_names=names,feature_types=schema[\'types\'],enable_categorical=True,nthread=args.threads))\n    else:\n        scores=model.predict(values,num_threads=args.threads)\n    predictions=rank(scores,pairs,targets)\n    score,ap=map_at_12(predictions,targets.article_id)\n    if args.stage.startswith(\'replay_\'):\n        assert np.array_equal(predictions,np.load(destination/f\'{split}_predictions.npy\'))\n        assert np.array_equal(ap,np.load(destination/f\'{split}_user_ap.npy\'))\n    else:\n        np.save(destination/f\'{split}_predictions.npy\',predictions)\n        np.save(destination/f\'{split}_user_ap.npy\',ap)\n    metrics={\'map\':score,\'users\':len(targets),\'pairs\':len(pairs),\'seconds\':time.perf_counter()-tick,\n             \'program_sha256\':hashlib.sha256(source.encode()).hexdigest(),\n             \'model_sha256\':hashlib.sha256(model_path.read_bytes()).hexdigest(),\'backend\':args.backend,\n             \'stage\':args.stage,\'threads\':args.threads,\'passed\':True,\'features\':names}\n    (destination/f\'{args.stage}_metrics.json\').write_text(json.dumps(metrics,indent=2))\n    print(json.dumps(metrics))\n\n\nif __name__==\'__main__\':\n    main()\n', 'runtime.py': '"""Notebook host: paper-style islands, local generation callback and CPU evaluator."""\nimport ast\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\nimport time\nimport zipfile\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom worker import RAW, check_program, Data, map_at_12\nfrom flat_features import SEED, RAW_ONLY, normalize\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef write_json(path,obj):\n    path=Path(path)\n    temp=path.with_suffix(path.suffix+\'.tmp\')\n    temp.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    temp.replace(path)\n\n\ndef locate_bundle(input_root,working,expected_manifest_sha):\n    input_root=Path(input_root)\n    matches=[]\n    for path in input_root.rglob(\'manifest.json\'):\n        try:\n            if json.loads(path.read_text()).get(\'bundle_id\')==\'hm_llmfe_kaggle_fulltrain_v2\':\n                matches.append(path.parent)\n        except (ValueError,OSError):\n            pass\n    if not matches:\n        archives=list(input_root.rglob(\'hm_llmfe_kaggle_fulltrain_v2.zip\'))\n        if len(archives)!=1:\n            raise FileNotFoundError(\'Add the supplied hm_llmfe_kaggle_fulltrain_v2.zip as a Kaggle Dataset/Input\')\n        destination=Path(working)/\'hm_llmfe_input\'\n        destination.mkdir(parents=True,exist_ok=True)\n        with zipfile.ZipFile(archives[0]) as z:\n            for member in z.infolist():\n                if not (destination/member.filename).resolve().is_relative_to(destination.resolve()):\n                    raise ValueError(\'Archive path outside destination\')\n            z.extractall(destination)\n        matches=[destination]\n    if len(matches)!=1:\n        raise ValueError(\'Several H&M bundles attached; keep exactly one\')\n    root=matches[0]\n    if sha(root/\'manifest.json\')!=expected_manifest_sha:\n        raise ValueError(\'Unexpected bundle manifest/version\')\n    manifest=json.loads((root/\'manifest.json\').read_text())\n    for name,info in manifest[\'files\'].items():\n        if sha(root/name)!=info[\'sha256\']:\n            raise ValueError(\'Data checksum mismatch: \'+name)\n    return root,manifest\n\n\ndef evaluate(data,directory,stage,config):\n    directory=Path(directory)\n    backend=config.get(\'backend\',config.get(\'search_backend\',\'lightgbm\'))\n    result=directory/f\'{stage}_metrics.json\'\n    if result.exists():\n        metrics=json.loads(result.read_text())\n        if metrics.get(\'backend\')!=backend:\n            raise RuntimeError(\'Cached result uses another evaluator; start a new run ID\')\n        if stage==\'preflight\':\n            if metrics[\'program_sha256\']!=sha(directory/\'program.py\'):\n                raise RuntimeError(\'Cached preflight uses another program\')\n            return metrics\n        model_path=directory/(\'model.txt\' if metrics.get(\'backend\')==\'lightgbm\' else \'model.json\')\n        if metrics[\'program_sha256\']!=sha(directory/\'program.py\') or metrics[\'model_sha256\']!=sha(model_path):\n            raise RuntimeError(\'Cached result does not match model/program\')\n        return metrics\n    command=[sys.executable,str(Path(__file__).with_name(\'worker.py\')),\'--data\',str(data),\n             \'--directory\',str(directory),\'--stage\',stage,\'--threads\',str(1 if stage.startswith(\'replay_\') else config[\'cpu_threads\']),\n             \'--memory-gb\',str(config[\'evaluation_memory_gb\']),\'--backend\',backend]\n    env={k:v for k,v in os.environ.items() if not any(s in k.upper() for s in [\'TOKEN\',\'SECRET\',\'API_KEY\'])}\n    env.update(CUDA_VISIBLE_DEVICES=\'\',OMP_NUM_THREADS=\'1\',OPENBLAS_NUM_THREADS=\'1\',MKL_NUM_THREADS=\'1\',NUMEXPR_NUM_THREADS=\'1\')\n    with (directory/f\'{stage}.log\').open(\'w\') as log:\n        try:\n            process=subprocess.run(command,env=env,stdout=log,stderr=subprocess.STDOUT,timeout=config[\'evaluation_timeout_seconds\'])\n        except subprocess.TimeoutExpired:\n            raise TimeoutError(f"Evaluation exceeded {config[\'evaluation_timeout_seconds\']} seconds") from None\n    if process.returncode or not result.exists():\n        raise RuntimeError((directory/f\'{stage}.log\').read_text()[-2000:])\n    return json.loads(result.read_text())\n\n\ndef source_from_response(text):\n    code=normalize(text)\n    check_program(code)\n    return code\n\n\n\ndef source_key(source):\n    tree=ast.parse(source)\n    for node in ast.walk(tree):\n        if isinstance(node,ast.FunctionDef) and node.body and isinstance(node.body[0],ast.Expr) and isinstance(node.body[0].value,ast.Constant) and isinstance(node.body[0].value.value,str):\n            node.body=node.body[1:]\n    return hashlib.sha256(ast.dump(tree,include_attributes=False).encode()).hexdigest()\n\n\ndef choose_demonstrations(memory,rng):\n    clusters={}\n    for item in memory:\n        clusters.setdefault(item[\'score\'],[]).append(item)\n    scores=np.array(list(clusters),dtype=float)\n    temperature=.1*(1-(len(memory)%10000)/10000)\n    weights=np.exp((scores-scores.max())/temperature)\n    weights/=weights.sum()\n    count=min(3,len(clusters))\n    selected=rng.choice(len(scores),size=count,replace=False,p=weights)\n    return [clusters[scores[i]][int(rng.integers(len(clusters[scores[i]])))] for i in selected]\n\n\ndef prompt(data,memory,seed):\n    dataset=Data(data)\n    pairs=pd.read_parquet(Path(data)/\'train_pairs.parquet\')\n    examples=pairs[pairs.timestamp==pairs.timestamp.max()].sample(5,random_state=42)\n    X=dataset.inputs(examples)\n    demos=choose_demonstrations(memory,np.random.default_rng(seed))\n    prior=[{\'search_MAP_fraction\':t[\'score\'],\'program\':(Path(t[\'directory\'])/\'program.py\').read_text()} for t in demos]\n    return \'\'\'You are a data scientist creating useful features for H&M clothing recommendations.\nTask: rank each customer\'s fixed candidate articles for purchases in the next7 days. Maximize validation MAP@12.\nComplete ONE Python function: def modify_features(X): return a numeric pandas DataFrame.\nX is already a prepared flat numeric table: one row per customer-article pair. No history joins needed.\npd (pandas) and np (numpy) are supplied; no other libraries, files, network or fitting.\nUse the existing columns below to create meaningful interactions, ratios, recency/popularity transforms,\nand affinities between customer preferences and candidate attributes. Use domain knowledge.\nStart from X.copy(), prefer preserving informative columns; you may drop less useful columns as in the examples.\nReturn a DataFrame with exactly X.index and row order. Numeric output1-78 columns, same schema for every call.\nUse vectorized ROWWISE operations only. No groupby, merge, joins, windows, random, ranking across rows,\nor dataset statistics such as column.mean(). No learning from the current batch. Missing values are allowed.\nYour function is applied in separate chunks and must give identical results regardless of other rows.\nFor ratios, handle zero denominators; np.log1p on nonnegative counts is supported.\nKeep the entire function compact, ideally2-6 new transformations, one short docstring, under2048 output tokens.\nReturn ONLY the complete function in a python code block; do not generate several versions or explanations.\n\nColumn meanings (all supplied by host, NOT invented by Llama):\nraw_u_age: customer age. raw_u_fn/raw_u_active: account flags.\nraw_u_club_member_status/raw_u_fashion_news_frequency: category codes.\nraw_i_*: encoded clothing product/type, graphic, colour, department, index, section, garment categories.\nCategory codes are identifiers, not continuous measurements; avoid arbitrary arithmetic on them.\nh_user_count_7/30/180 and h_article_count_7/30/180: past purchase counts in those calendar-day windows.\nh_user_recency/h_article_recency: days since most recent purchase in past180 days.\nh_user_price_mean/h_article_price_mean: average normalized past price in past180 days.\nh_user_type_count_30/180: customer\'s purchases of candidate\'s product_type_no in those windows.\nh_user_article_count/h_user_article_recency: past180-day count/recency for this exact customer-article.\nh_user_colour_count: customer\'s past180-day purchases of candidate\'s colour group.\nMissing aggregates mean no matching history; counts may be filled with0 but missing recency is NOT recent.\nHistory is restricted to selected customer cohort and180 days before each cutoff, not population-wide sales.\nOnly30 base columns exist. Do not invent other history fields, IDs, timestamps, labels or prices of future purchases.\nThe host scores all803377 supervised training pairs with CPU LightGBM150 rounds/31 leaves, seed42.\nSearch uses fixed256 validation users and6594 candidates. Test is unavailable during selection.\nPreviously measured complete transformations include a real working seeded example.\n\'\'\'+json.dumps({\'available_columns\':list(X),\'five_training_examples\':X.to_dict(\'records\'),\n               \'working_host_seed_example\':SEED,\n               \'previous_programs\':prior},separators=(\',\',\':\'),default=str)\n\n\n\ndef budget_summary(trials,config):\n    measured=sum(t[\'status\']==\'valid\' for t in trials)\n    return {\'llm_calls\':len(trials),\'repair_calls\':sum(t.get(\'repair\',0)>0 for t in trials),\n            \'new_proposals\':len({t[\'proposal\'] for t in trials}),\n            \'rejected\':sum(t[\'status\']==\'rejected\' for t in trials),\n            \'duplicates\':sum(t[\'status\']==\'duplicate\' for t in trials),\n            \'evaluated_distinct\':measured,\'target\':config[\'target_successful_distinct\'],\n            \'max_llm_calls\':config[\'max_llm_calls\'],\'max_repairs_per_proposal\':config[\'max_repairs_per_proposal\'],\n            \'target_met\':measured>=config[\'target_successful_distinct\'],\n            \'budget_exhausted\':len(trials)>=config[\'max_llm_calls\'],\n            \'prompt_tokens\':sum(t.get(\'prompt_tokens\',0) for t in trials),\n            \'llm_seconds\':sum(t.get(\'llm_seconds\',0) for t in trials),\n            \'generated_tokens\':sum(t.get(\'generated_tokens\',0) for t in trials)}\n\n\ndef progress(out,trials,config):\n    out=Path(out)\n    write_json(out/\'trials.json\',trials)\n    summary=budget_summary(trials,config)\n    write_json(out/\'search_state.json\',summary)\n    valid=[t for t in trials if t[\'status\']==\'valid\']\n    fig,axes=plt.subplots(1,3,figsize=(15,4))\n    if valid:\n        x=[t[\'sample\'] for t in valid]\n        y=100*np.array([t[\'score\'] for t in valid])\n        axes[0].scatter(x,y,label=\'Candidate\')\n        axes[0].step(x,np.maximum.accumulate(y),where=\'post\',label=\'Best search\')\n        axes[0].legend()\n        axes[1].plot(np.arange(1,len(valid)+1),np.maximum.accumulate(y),marker=\'.\')\n        axes[2].hist(y,bins=min(10,len(valid)))\n    axes[0].set(xlabel=\'LLM call (repairs included)\',ylabel=\'Search MAP@12 (%)\')\n    axes[1].set(xlabel=\'Distinct evaluated sets\',ylabel=\'Best search MAP@12 (%)\')\n    axes[2].set(xlabel=\'Search MAP@12 (%)\',ylabel=\'Evaluated programs\')\n    fig.suptitle(\'Search only: fixed256 users/candidates — not test\')\n    fig.tight_layout();fig.savefig(out/\'search_progress.png\',dpi=150);plt.close(fig)\n    fig,axes=plt.subplots(1,2,figsize=(11,4))\n    labels=[\'valid\',\'rejected\',\'duplicate\']\n    axes[0].bar(labels,[sum(t[\'status\']==s for t in trials) for s in labels])\n    axes[0].set_ylabel(\'Generated programs\')\n    for t in valid:\n        p=Path(t[\'directory\'])/\'learning_curve.json\'\n        history=json.loads(p.read_text())[\'train\'][\'logloss\']\n        axes[1].plot(np.arange(1,len(history)+1),history,label=str(t[\'sample\']),alpha=.7)\n    axes[1].set(xlabel=\'Boosting round\',ylabel=\'Train binary logloss\')\n    if valid:\n        axes[1].legend(ncol=4,fontsize=7)\n    fig.tight_layout();fig.savefig(out/\'training_and_rejections.png\',dpi=150);plt.close(fig)\n    lines=[\'# Search progress\',\'\',\n           f"LLM calls: {len(trials)}/{summary[\'max_llm_calls\']}; evaluated distinct: {len(valid)}/{summary[\'target\']}; repair calls: {summary[\'repair_calls\']}.",\n           \'Search validation only; no final test results yet.\',\'\',\n           \'| Call | Proposal | Island | Repair | Parent call | Status | MAP@12, % |\',\'|---|---:|---:|---:|---:|---|---:|\']\n    for t in trials:\n        value=f"{100*t[\'score\']:.5f}" if t[\'status\']==\'valid\' else \'—\'\n        lines.append(f"| {t[\'sample\']} | {t[\'proposal\']} | {t[\'island\']} | {t[\'repair\']} | {t.get(\'parent_call\') or \'—\'} | {t[\'status\']} | {value} |")\n    lines+=[\'\',\'![Search](search_progress.png)\',\'![Training](training_and_rejections.png)\']\n    (out/\'progress.md\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\n\n\ndef repair_prompt(base,parent,error):\n    return (base+\'\\nFix the previous program. Return ONE COMPLETE corrected function, not a patch.\\n\'\n            +\'Previous response:\\n\'+parent+\'\\nRejection:\\n\'+error[-3000:]\n            +\'\\nUse only the listed flat columns. No history/groupby/joins. Preserve row index and schema. \'\n            +\'If duplicate choose different useful transformations. Keep code short.\')\n\n\n\ndef search(data,out,config,generate):\n    data,out=Path(data),Path(out)\n    out.mkdir(parents=True,exist_ok=True)\n    if (out/\'selected.json\').exists():\n        return json.loads((out/\'selected.json\').read_text())\n    control=out/\'raw_control\';control.mkdir(exist_ok=True)\n    (control/\'program.py\').write_text(RAW)\n    baseline=evaluate(data,control,\'search\',config)\n    raw={\'directory\':str(control),\'score\':baseline[\'map\']}\n    seed_dir=out/\'seed_control\';seed_dir.mkdir(exist_ok=True)\n    (seed_dir/\'program.py\').write_text(SEED)\n    seed_score=evaluate(data,seed_dir,\'search\',config)\n    seed={\'directory\':str(seed_dir),\'score\':seed_score[\'map\']}\n    raw15_dir=out/\'raw15_control\';raw15_dir.mkdir(exist_ok=True)\n    (raw15_dir/\'program.py\').write_text(RAW_ONLY)\n    raw15_score=evaluate(data,raw15_dir,\'search\',config)\n    raw15={\'directory\':str(raw15_dir),\'score\':raw15_score[\'map\']}\n    memories=[[raw.copy(),seed.copy()] for _ in range(3)]\n    trials=[]\n    seen=set()\n    target=config[\'target_successful_distinct\']\n    max_calls=config[\'max_llm_calls\']\n    max_repairs=config[\'max_repairs_per_proposal\']\n    if target<1 or max_calls<1 or max_repairs<0:\n        raise ValueError(\'Positive successful-set/request budgets and nonnegative repair budget required\')\n    proposal=0\n    repair=0\n    parent=None\n    while len(seen)<target and len(trials)<max_calls:\n        index=len(trials)\n        batch=proposal//3\n        batchdir=out/f\'batch_{batch:02d}\';batchdir.mkdir(exist_ok=True)\n        if not (batchdir/\'meta.json\').exists():\n            island=int(np.random.default_rng(42+batch).integers(3))\n            text=prompt(data,memories[island],42+batch)\n            (batchdir/\'prompt.txt\').write_text(text)\n            write_json(batchdir/\'meta.json\',{\'island\':island,\'new_proposals_per_prompt\':3,\'seed\':42+batch})\n        meta=json.loads((batchdir/\'meta.json\').read_text());island=meta[\'island\']\n        directory=out/\'candidates\'/f\'candidate_{index+1:03d}\';directory.mkdir(parents=True,exist_ok=True)\n        if (directory/\'trial.json\').exists():\n            trial=json.loads((directory/\'trial.json\').read_text())\n        else:\n            trial={\'sample\':index+1,\'call\':index+1,\'proposal\':proposal+1,\'repair\':repair,\n                   \'parent_call\':parent[\'sample\'] if parent is not None else None,\n                   \'island\':island,\'batch\':batch,\'directory\':str(directory)}\n            tick=time.perf_counter()\n            failure_stage=\'generation\'\n            try:\n                response_path=directory/\'response.json\'\n                text=(batchdir/\'prompt.txt\').read_text()\n                if repair:\n                    parent_path=Path(parent[\'directory\'])/\'response.json\'\n                    previous=json.loads(parent_path.read_text()).get(\'text\',\'\') if parent_path.exists() else \'\'\n                    # Keep repair prompts within context: prior code + error + raw example,\n                    # rather than appending to three potentially long memory programs.\n                    text=repair_prompt(prompt(data,[seed],42+proposal),previous,parent[\'error\'])\n                (directory/\'prompt.txt\').write_text(text)\n                if response_path.exists():\n                    response=json.loads(response_path.read_text())\n                else:\n                    request_path=directory/\'request.json\'\n                    if request_path.exists():\n                        raise RuntimeError(\'Previous request was interrupted; its budget slot is consumed. Use a new request for repair.\')\n                    write_json(request_path,{**trial,\'seed\':42+index})\n                    print(f"LLM call{index+1}/{max_calls}, measured{len(seen)}/{target}, island{island}, repair{repair}/{max_repairs}",flush=True)\n                    response=generate(text,42+index)\n                    write_json(response_path,response)\n                for name in [\'prompt_tokens\',\'generated_tokens\',\'seconds\']:\n                    trial[\'llm_seconds\' if name==\'seconds\' else name]=response.get(name,0)\n                trial[\'truncated\']=bool(response.get(\'truncated\'))\n                failure_stage=\'parse_and_validate\'\n                if response.get(\'truncated\'):\n                    raise ValueError(\'Program truncated at output token limit\')\n                code=source_from_response(response[\'text\'])\n                (directory/\'program.py\').write_text(code)\n                key=source_key(code);trial[\'program_key\']=key\n                if key in seen or key in {source_key(RAW),source_key(SEED),source_key(RAW_ONLY)}:\n                    trial.update(status=\'duplicate\',error=\'Duplicate Python AST\')\n                else:\n                    failure_stage=\'preflight\'\n                    print(f"call{index+1}: Python validation passed; checking small time-safe snapshots (not a measured set)",flush=True)\n                    preflight=evaluate(data,directory,\'preflight\',config)\n                    trial[\'preflight\']=preflight\n                    failure_stage=\'feature_train_score\'\n                    print(f"call{index+1}: preflight passed; starting LightGBM on{config.get(\'train_pairs\',\'configured\')} training pairs",flush=True)\n                    metrics=evaluate(data,directory,\'search\',config)\n                    trial.update(status=\'valid\',score=metrics[\'map\'],metrics=metrics)\n            except Exception as error:\n                trial.update(status=\'rejected\',error=str(error),failure_stage=failure_stage)\n            trial[\'elapsed_seconds\']=time.perf_counter()-tick\n            write_json(directory/\'trial.json\',trial)\n        trials.append(trial)\n        if trial[\'status\']==\'valid\':\n            seen.add(trial[\'program_key\'])\n            if trial[\'score\']>max(t[\'score\'] for t in memories[island]):\n                memories[island].append(trial)\n        write_json(out/\'memory.json\',memories)\n        progress(out,trials,config)\n        print(f"call{index+1}/{max_calls}, island{island}, repair{repair}: {trial[\'status\']}; measured={len(seen)}/{target}",flush=True)\n        if trial[\'status\']!=\'valid\':\n            print(f"REJECTION call{index+1}: stage={trial.get(\'failure_stage\',\'duplicate\')}, "\n                  f"generated_tokens={trial.get(\'generated_tokens\')}, truncated={trial.get(\'truncated\')}",flush=True)\n            print(trial.get(\'error\',\'No error text saved\'),flush=True)\n        else:\n            print(f"ACCEPTED: search MAP@12={100*trial[\'score\']:.6f}%",flush=True)\n        if len(trials)>=config.get(\'abort_if_no_success_after_calls\',6) and not seen:\n            raise RuntimeError(\'Pilot gate: no successfully measured LLM programs; stopping before spending300 calls\')\n        if trial[\'status\']!=\'valid\' and repair<max_repairs:\n            repair+=1\n            parent=trial\n        else:\n            proposal+=1\n            repair=0\n            parent=None\n    write_json(out/\'search_complete.json\',budget_summary(trials,config))\n    # Raw control can win; selecting a harmful LLM program is not forced.\n    options=[raw,seed,raw15]+[t for t in trials if t[\'status\']==\'valid\']\n    options=sorted(options,key=lambda t:t[\'score\'],reverse=True)[:3]\n    confirmations=[]\n    for t in options:\n        metrics=evaluate(data,Path(t[\'directory\']),\'confirm\',config)\n        confirmations.append({\'directory\':t[\'directory\'],\'score\':metrics[\'map\'],\'metrics\':metrics})\n        write_json(out/\'confirmation.json\',confirmations)\n    best=max(confirmations,key=lambda t:t[\'score\'])\n    selected={**best,\'selection\':\'512 disjoint validation users; frozen before heldout val/test\',\n              \'selected_raw_control\':Path(best[\'directory\']) in {control,seed_dir,raw15_dir},\n              \'selected_seed_control\':Path(best[\'directory\'])==seed_dir,\n              \'search_budget\':budget_summary(trials,config)}\n    write_json(out/\'selected.json\',selected)\n    return selected\n\n\ndef final(data,out,config):\n    data,out=Path(data),Path(out)\n    selected=json.loads((out/\'selected.json\').read_text())\n    winner_name=\'Search winner: host control (NO LLM features)\' if selected[\'selected_raw_control\'] else \'LLM-FE flat LightGBM\'\n    choices={\'Raw15 LightGBM\':out/\'raw15_control\',\'Prepared30 LightGBM\':out/\'raw_control\',\'Seeded32 LightGBM\':out/\'seed_control\',winner_name:Path(selected[\'directory\'])}\n    metrics={}\n    for name,directory in choices.items():\n        settings={**config,\'backend\':\'lightgbm\'}\n        metrics[name]={}\n        for split in [\'val\',\'test\']:\n            original=evaluate(data,directory,split,settings)\n            replay=evaluate(data,directory,\'replay_\'+split,settings)\n            assert original[\'map\']==replay[\'map\'] and replay[\'passed\']\n            metrics[name][split]=original\n    # Native reference is never included in prompts or model selection.\n    references={\'Manual features → LightGBM\':\'manual_lightgbm\',\n                \'GNN → LightGBM\':\'gnn_lightgbm\',\n                \'ID-GNN (graph training budget)\':\'idgnn\'}\n    for name,label in references.items():\n        metrics[name]={}\n        for split in [\'val\',\'test\']:\n            targets=pd.read_parquet(data/f\'{split}_targets.parquet\')\n            value,_=map_at_12(np.load(data/f\'{split}_{label}_reference.npy\'),targets.article_id)\n            metrics[name][split]={\'map\':value,\'reference\':True}\n    frame=pd.DataFrame([{\'model\':name,\'val_map\':r[\'val\'][\'map\'],\'test_map\':r[\'test\'][\'map\']} for name,r in metrics.items()])\n    frame.to_csv(out/\'comparison.csv\',index=False)\n    fig,ax=plt.subplots(figsize=(10,4))\n    for i,split in enumerate([\'val\',\'test\']):\n        bars=ax.barh(np.arange(len(frame))+(i-.5)*.32,100*frame[split+\'_map\'],height=.32,label=split)\n        ax.bar_label(bars,fmt=\'%.3f\',padding=3,fontsize=8)\n    ax.set_yticks(np.arange(len(frame)),frame.model);ax.set_xlabel(\'MAP@12 (%) — identical fixed candidate pools\');ax.legend()\n    ax.set_xlim(0,max(.1,float(frame[[\'val_map\',\'test_map\']].to_numpy().max())*118))\n    if config.get(\'smoke\'):\n        fig.suptitle(\'SYNTHETIC SMOKE — not Llama results\')\n    fig.tight_layout();fig.savefig(out/\'comparison.png\',dpi=150,bbox_inches=\'tight\');plt.close(fig)\n    aps={name:np.load(path/\'test_user_ap.npy\') for name,path in choices.items()}\n    truth=pd.read_parquet(data/\'test_targets.parquet\')\n    for name,label in references.items():\n        aps[name]=map_at_12(np.load(data/f\'test_{label}_reference.npy\'),truth.article_id)[1]\n    pd.DataFrame({\'customer_id\':truth.customer_id.values,**aps}).to_csv(out/\'test_user_ap.csv\',index=False)\n    fig,axes=plt.subplots(1,2,figsize=(13,4))\n    for name,values in aps.items():\n        axes[0].hist(100*values[values>0],bins=np.linspace(0,100,21),histtype=\'step\',label=name)\n    axes[0].set(xlabel=\'Positive user AP@12 (%)\',ylabel=\'Users\',yscale=\'log\');axes[0].legend(fontsize=7)\n    axes[1].barh(list(aps),[100*np.mean(a==0) for a in aps.values()]);axes[1].set(xlabel=\'Test users without hits (%)\',xlim=(0,100))\n    if config.get(\'smoke\'):\n        fig.suptitle(\'SYNTHETIC SMOKE — not Llama results\')\n    fig.tight_layout();fig.savefig(out/\'ap_histograms.png\',dpi=150,bbox_inches=\'tight\');plt.close(fig)\n    delta=aps[winner_name]-aps[\'Prepared30 LightGBM\']\n    fig,axes=plt.subplots(1,2,figsize=(11,4))\n    axes[0].bar([\'Winner better\',\'Tie\',\'Prepared30 better\'],[(delta>0).sum(),(delta==0).sum(),(delta<0).sum()])\n    if np.any(delta!=0):\n        axes[1].hist(100*delta[delta!=0],bins=30);axes[1].set_yscale(\'log\')\n    axes[1].set(xlabel=\'LLM minus prepared30 AP (percentage points)\',ylabel=\'Users; ties omitted\')\n    fig.tight_layout();fig.savefig(out/\'paired_comparison.png\',dpi=150);plt.close(fig)\n    fig,axes=plt.subplots(1,2,figsize=(11,4))\n    for name in [\'Prepared30 LightGBM\',winner_name]:\n        curve=json.loads((choices[name]/\'learning_curve.json\').read_text())[\'train\'][\'logloss\']\n        axes[0].plot(np.arange(1,len(curve)+1),curve,label=name)\n    axes[0].set(xlabel=\'Boosting round\',ylabel=\'Train binary logloss\');axes[0].legend()\n    delta_lgb=aps[winner_name]-aps[\'Prepared30 LightGBM\']\n    axes[1].bar([\'Winner better\',\'Tie\',\'Prepared30 better\'],[(delta_lgb>0).sum(),(delta_lgb==0).sum(),(delta_lgb<0).sum()])\n    axes[1].set_title(\'LightGBM: paired test AP\')\n    fig.tight_layout();fig.savefig(out/\'lightgbm_comparison.png\',dpi=150);plt.close(fig)\n    trials=json.loads((out/\'trials.json\').read_text())\n    report={\'complete\':True,\'metrics\':metrics,\'selected\':selected,\'config\':config,\n            \'search_budget\':budget_summary(trials,config),\n            \'smoke\':bool(config.get(\'smoke\',False)),\n            \'generated_programs\':sum((Path(t[\'directory\'])/\'response.json\').exists() for t in trials),\n            \'evaluated_distinct\':sum(t[\'status\']==\'valid\' for t in trials),\n            \'llm_feature_model_selected\':not selected[\'selected_raw_control\'],\n            \'llm_program_quality_measured\':any(t[\'status\']==\'valid\' for t in trials),\n            \'independent_full_replay\':True,\'coverage\':\'bounded exported subset, not entire H&M\',\n            \'paper_reproduction\':False,\'base_feature_origin\':\'15raw+15human historical aggregates\',\'seed_control_origin\':\'2human log transforms\'}\n    write_json(out/\'report.json\',report)\n    lines=[\'# Llama3.1-8B → Python features → LightGBM: bounded H&M\',\'\',\n           \'Flat-table adaptation:15 raw fields+15 human historical aggregates;2 manual seed transforms. Not all features are LLM-generated. All MAP values in percent.\',\n           \'One seed42,180-day selected-cohort history, frozen shared ID-GNN pools.\',\n           \'LightGBM selects features throughout search and confirmation; all new trees use identical803377 training pairs.\',\n           \'Manual and GNN→LightGBM references use those same803377 supervised pairs and these exact evaluation rows/pools.\',\n           \'Tuning protocols, derived seeds and historical representations differ; GNN also used upstream graph training.\',\n           \'Compare pipeline quality on this subset, not differences in isolated feature quality or full-dataset MAP.\',\n           \'Selection uses only search256 + confirmation512; heldout val1024 and test2048 used after freezing.\',\n           \'Native ID-GNN reference used more training data and is contextual, not a matched-budget comparison.\',\n           f"LLM calls{len(trials)}/{config[\'max_llm_calls\']}; evaluated distinct{report[\'evaluated_distinct\']}/{config[\'target_successful_distinct\']}; target met: {report[\'search_budget\'][\'target_met\']}. Raw control selected: {selected[\'selected_raw_control\']}.",\n           \'All raw/LLM heldout predictions reproduced from saved models in separate CPU processes with1 thread.\',\n           \'\', \'| Model | Validation MAP, % | Test MAP, % |\',\'|---|---:|---:|\']\n    if config.get(\'smoke\'):\n        lines=[\'# SYNTHETIC SMOKE — not Llama results\',\'\',\n               \'This report tests software using a synthetic generator and tiny data. No real Llama inference or quality experiment.\', \'\']+lines\n    if selected[\'selected_raw_control\']:\n        lines.insert(2,\'RAW CONTROL SELECTED: the search-winner row contains NO LLM features. Do not report its MAP as the quality of LLM-generated features.\')\n    lines += [f"| {r.model} | {100*r.val_map:.5f} | {100*r.test_map:.5f} |" for r in frame.itertuples()]\n    lines += [\'\',f"Selected program: `{Path(selected[\'directory\']).relative_to(out)}/program.py`.",\n              \'![MAP](comparison.png)\',\'![AP](ap_histograms.png)\',\'![Paired](paired_comparison.png)\',\n              \'![Search](search_progress.png)\',\'![Training](training_and_rejections.png)\',\n              \'![LightGBM](lightgbm_comparison.png)\']\n    (out/\'report.md\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\n    return report\n', 'flat_features.py': '"""Fixed past-only base table; Llama only transforms existing numeric columns."""\nimport ast\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport signal\nimport numpy as np\nimport pandas as pd\nfrom sql_features import compile_sql\nfrom original_data import Data as HistoryData\nfrom program_guard import extract\n\nRAW=\'def modify_features(X):\\n    return X.copy()\\n\'\nSEED=\'\'\'def modify_features(X):\n    F = X.copy()\n    F[\'f_log_user_purchases\'] = np.log1p(X[\'h_user_count_180\'].fillna(0))\n    F[\'f_log_article_purchases\'] = np.log1p(X[\'h_article_count_30\'].fillna(0))\n    return F\n\'\'\'\nRAW_ONLY=\'def modify_features(X):\\n    return X[[c for c in X.columns if c.startswith("raw_")]].copy()\\n\'\nBASE_SPEC={\'aggregates\':[], \'derived\':[]}\nfor entity,key in [(\'user\',\'customer_id\'),(\'article\',\'article_id\')]:\n    for window in [7,30,180]:\n        BASE_SPEC[\'aggregates\'].append({\'name\':f\'f_{entity}_count_{window}\',\'keys\':[key],\n            \'sql\':f\'COUNT(*) FILTER (WHERE days_ago <= {window})\'})\n    BASE_SPEC[\'aggregates\'] += [\n        {\'name\':f\'f_{entity}_recency\',\'keys\':[key],\'sql\':\'MIN(days_ago)\'},\n        {\'name\':f\'f_{entity}_price_mean\',\'keys\':[key],\'sql\':\'AVG(price)\'}]\nfor window in [30,180]:\n    BASE_SPEC[\'aggregates\'].append({\'name\':f\'f_user_type_count_{window}\',\n        \'keys\':[\'customer_id\',\'product_type_no\'],\'sql\':f\'COUNT(*) FILTER (WHERE days_ago <= {window})\'})\nBASE_SPEC[\'aggregates\'] += [\n    {\'name\':\'f_user_article_count\',\'keys\':[\'customer_id\',\'article_id\'],\'sql\':\'COUNT(*)\'},\n    {\'name\':\'f_user_article_recency\',\'keys\':[\'customer_id\',\'article_id\'],\'sql\':\'MIN(days_ago)\'},\n    {\'name\':\'f_user_colour_count\',\'keys\':[\'customer_id\',\'colour_group_code\'],\'sql\':\'COUNT(*)\'}]\n\n\ndef prepare(source,working):\n    source=Path(source);working=Path(working)\n    source_manifest=json.loads((source/\'manifest.json\').read_text())\n    identity={\'source_manifest_sha256\':hashlib.sha256((source/\'manifest.json\').read_bytes()).hexdigest(),\n              \'fixed_base_spec\':BASE_SPEC,\'preparer_sha256\':hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n              \'history_data_sha256\':hashlib.sha256(Path(__file__).with_name(\'original_data.py\').read_bytes()).hexdigest(),\n              \'sql_engine_sha256\':hashlib.sha256(Path(__file__).with_name(\'sql_features.py\').read_bytes()).hexdigest()}\n    key=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()[:12]\n    root=working/(\'hm_flat_table_v1_\'+key);root.mkdir(parents=True,exist_ok=True)\n    done=root/\'flat_manifest.json\'\n    if done.exists():\n        manifest=json.loads(done.read_text())\n        assert manifest[\'identity\']==identity\n        for name,digest in manifest[\'file_sha256\'].items():\n            assert hashlib.sha256((root/name).read_bytes()).hexdigest()==digest,name\n        return root,manifest\n    data=HistoryData(source);function=compile_sql(json.dumps(BASE_SPEC))\n    fields=None;files=[]\n    for split in [\'train\',\'search\',\'confirm\',\'val\',\'test\']:\n        pairs=pd.read_parquet(source/f\'{split}_pairs.parquet\')\n        frames=[]\n        for cutoff,frame in pairs.groupby(\'timestamp\',sort=False):\n            X,history=data.inputs(frame,cutoff)\n            F=function(X,history).rename(columns=lambda c:\'h_\'+c[2:])\n            flat=pd.concat([X[data.raw_u+data.raw_i],F],axis=1).astype(\'float32\')\n            if fields is None:fields=list(flat)\n            assert list(flat)==fields\n            flat.index=frame.index;frames.append(flat)\n            print(\'Preparing flat table:\',split,str(cutoff),\'pairs\',len(frame),flush=True)\n        flat=pd.concat(frames).reindex(pairs.index)\n        assert len(flat)==len(pairs) and flat.index.equals(pairs.index)\n        tagged=pairs.copy();tagged[\'_flat_row\']=np.arange(len(pairs));tagged[\'_flat_split\']=split\n        pd.testing.assert_frame_equal(tagged[pairs.columns],pairs)\n        for name,frame in [(f\'{split}_pairs.parquet\',tagged),(f\'{split}_flat.parquet\',flat)]:\n            temp=root/(name+\'.tmp\');frame.to_parquet(temp,index=False);temp.replace(root/name);files.append(name)\n    for path in source.iterdir():\n        if path.name.endswith(\'_targets.parquet\') or path.name.endswith(\'_reference.npy\'):\n            shutil.copy2(path,root/path.name);files.append(path.name)\n    copied_manifest={**source_manifest,\'flat_base_fields\':fields,\'source_pair_rows_preserved\':True}\n    (root/\'manifest.json\').write_text(json.dumps(copied_manifest));files.append(\'manifest.json\')\n    manifest={\'identity\':identity,\'base_fields\':fields,\'category_maps\':data.category_maps,\n              \'raw_u\':data.raw_u,\'raw_i\':data.raw_i,\'train_pairs\':source_manifest[\'train_pairs\'],\n              \'base_feature_origin\':\'15 raw static +15 fixed human-defined past-only aggregates\',\n              \'file_sha256\':{name:hashlib.sha256((root/name).read_bytes()).hexdigest() for name in files}}\n    done.write_text(json.dumps(manifest,indent=2))\n    return root,manifest\n\n\ndef normalize(source):\n    return extract(source)\n\n\ndef check_program(source):\n    normalized=normalize(source)\n    tree=ast.parse(normalized)\n    blocked={\'groupby\',\'rolling\',\'expanding\',\'ewm\',\'pivot\',\'pivot_table\',\'merge\',\'join\',\'sample\',\n             \'fit\',\'fit_transform\',\'transform\',\'rank\',\'quantile\',\'value_counts\',\'corr\',\'cov\',\'describe\'}\n    reductions={\'mean\',\'sum\',\'min\',\'max\',\'median\',\'std\',\'var\',\'count\',\'nunique\'}\n    for node in ast.walk(tree):\n        if isinstance(node,ast.Call) and isinstance(node.func,ast.Attribute):\n            method=node.func.attr\n            if method in blocked:raise ValueError(\'Flat rowwise transformations only; forbidden: \'+method)\n            if method in reductions and not any(k.arg==\'axis\' and isinstance(k.value,ast.Constant) and k.value.value==1 for k in node.keywords):\n                raise ValueError(\'No statistics across rows: \'+method+\'; only axis=1 reductions are allowed\')\n    return normalized\n\n\ndef compile_program(source):\n    normalized=check_program(source)\n    import builtins\n    names=[\'abs\',\'min\',\'max\',\'sum\',\'len\',\'range\',\'enumerate\',\'zip\',\'list\',\'tuple\',\'dict\',\'set\',\n           \'float\',\'int\',\'str\',\'bool\',\'round\',\'isinstance\',\'any\',\'all\',\'ValueError\',\'Exception\']\n    scope={\'pd\':pd,\'np\':np,\'__builtins__\':{n:getattr(builtins,n) for n in names}}\n    exec(compile(normalized,\'<flat-features>\',\'exec\'),scope)\n    function=scope[\'modify_features\'];function._feature_source=source\n    return function\n\n\nclass Data:\n    def __init__(self,root):\n        self.root=Path(root)\n        self.manifest=json.loads((self.root/\'manifest.json\').read_text())\n        self.flat_manifest=json.loads((self.root/\'flat_manifest.json\').read_text())\n        self.fields=self.flat_manifest[\'base_fields\'];self.category_maps=self.flat_manifest[\'category_maps\']\n        # Keep all base fields in the baseline; only raw_i categories are categorical in LightGBM.\n        self.raw_u=self.flat_manifest[\'raw_u\']+[c for c in self.fields if c.startswith(\'h_\')]\n        self.raw_i=self.flat_manifest[\'raw_i\'];self.cache={}\n\n    def inputs(self,pairs):\n        split=pairs[\'_flat_split\'].iloc[0]\n        assert pairs[\'_flat_split\'].eq(split).all()\n        if split not in self.cache:self.cache[split]=pd.read_parquet(self.root/f\'{split}_flat.parquet\')\n        return self.cache[split].iloc[pairs[\'_flat_row\'].to_numpy()].reset_index(drop=True).copy()\n\n\ndef matrix(data,pairs,func,expected=None):\n    result=[];fields=None\n    raw_only=func._feature_source==RAW_ONLY\n    for offset in range(0,len(pairs),65536):\n        selected=pairs.iloc[offset:offset+65536]\n        X=data.inputs(selected)\n        def expired(*_):raise TimeoutError(\'Flat feature function exceeded30s\')\n        signal.signal(signal.SIGALRM,expired);signal.alarm(30)\n        try:out=func(X.copy(deep=True))\n        finally:signal.alarm(0)\n        if not isinstance(out,pd.DataFrame) or not out.index.equals(X.index) or out.columns.duplicated().any():\n            raise ValueError(\'Return DataFrame preserving X rows/index and unique columns\')\n        if not 1<=len(out.columns)<=len(data.fields)+48:raise ValueError(\'Require1-78 numeric feature columns\')\n        if any(not isinstance(c,str) for c in out):raise ValueError(\'Column names must be strings\')\n        changed=list(out)!=list(X) or any(not np.array_equal(out[c].to_numpy(),X[c].to_numpy(),equal_nan=True) for c in out if c in X)\n        func._feature_changed=getattr(func,\'_feature_changed\',False) or changed\n        if any(not pd.api.types.is_numeric_dtype(out[c]) for c in out):raise ValueError(\'All columns must be numeric\')\n        if fields is None:fields=list(out)\n        if list(out)!=fields or expected is not None and fields!=expected:raise ValueError(\'Feature schema changed\')\n        values=out.to_numpy(dtype=\'float32\');values[~np.isfinite(values)]=np.nan\n        result.append(values)\n    return np.concatenate(result),fields\n', 'original_data.py': "import json\nfrom pathlib import Path\nimport pandas as pd\nimport numpy as np\nclass Data:\n    def __init__(self,root):\n        self.root=Path(root)\n        self.manifest=json.loads((self.root/'manifest.json').read_text())\n        self.items=pd.read_parquet(self.root/'articles.parquet').set_index('article_id')\n        # XGBoost allocates categorical histograms up to the maximum code. Compact\n        # all static item categories consistently in both X and history.\n        self.category_maps={}\n        for column in self.items:\n            encoded,categories=pd.factorize(self.items[column],sort=True)\n            self.category_maps[column]=categories.tolist()\n            self.items[column]=encoded.astype('int32')\n        self.users=pd.read_parquet(self.root/'customers.parquet').set_index('customer_id')\n        self.tx=pd.read_parquet(self.root/'history.parquet')\n        self.raw_u=['raw_u_'+c.lower() for c in self.users.columns]\n        self.raw_i=['raw_i_'+c for c in self.items.columns]\n\n    def inputs(self,pairs,cutoff):\n        pairs=pairs[['customer_id','article_id']].reset_index(drop=True)\n        users=self.users.reindex(pairs.customer_id).reset_index(drop=True)\n        items=self.items.reindex(pairs.article_id).reset_index(drop=True)\n        users.columns=self.raw_u\n        items.columns=self.raw_i\n        X=pd.concat([pairs,users,items],axis=1)\n        cutoff=pd.Timestamp(cutoff)\n        history=self.tx[(self.tx.t_dat<=cutoff)&(self.tx.t_dat>cutoff-pd.Timedelta(days=180))].copy()\n        assert (history.t_dat<=cutoff).all()\n        history['days_ago']=(cutoff-history.t_dat).dt.days.astype('int32')\n        history=history.drop(columns='t_dat').merge(self.items.reset_index(),on='article_id',how='left',validate='many_to_one')\n        history=history.merge(self.users[['age']].rename(columns={'age':'customer_age'}).reset_index(),on='customer_id',how='left',validate='many_to_one')\n        return X,history\n\n", 'program_guard.py': "import ast\nimport re\nFORBIDDEN={'open','eval','exec','compile','getattr','setattr','globals','locals','vars','__import__',\n           'input','breakpoint','help','dir','type','super'}\n\ndef extract(text):\n    blocks=re.findall(r'```(?:python)?\\s*\\n(.*?)```',text,re.S|re.I)\n    source=next((b for b in blocks if 'def modify_features' in b),text).strip()\n    if not re.search(r'def modify_features(?:_v\\d+)?\\s*\\(',source):\n        # Authors sample a continuation after a prefilled function header.\n        source='def modify_features(df_input):\\n'+ '\\n'.join('    '+line for line in source.splitlines())\n    tree=ast.parse(source)\n    functions=[n for n in tree.body if isinstance(n,ast.FunctionDef) and re.fullmatch(r'modify_features(?:_v\\d+)?',n.name)]\n    if not functions:raise ValueError('No modify_features function')\n    function=functions[-1]\n    if len(function.args.args)!=1 or function.decorator_list:\n        raise ValueError('One dataframe argument, no decorators')\n    function.name='modify_features'\n    for node in ast.walk(function):\n        if isinstance(node,(ast.Import,ast.ImportFrom)):\n            permitted=isinstance(node,ast.Import) and all(a.name in {'pandas','numpy'} and a.asname=={'pandas':'pd','numpy':'np'}[a.name] for a in node.names)\n            if not permitted:raise ValueError('Only imports pandas as pd / numpy as np are supported')\n        if isinstance(node,(ast.Global,ast.Nonlocal,ast.ClassDef,ast.With)):\n            raise ValueError('No global state, classes or context managers')\n        if isinstance(node,ast.Name) and (node.id.startswith('_') or node.id in FORBIDDEN):raise ValueError('Forbidden name '+node.id)\n        if isinstance(node,ast.Attribute) and (node.attr.startswith('_') or node.attr.startswith(('read_','to_')) and node.attr not in {'to_numpy','to_frame','to_list','to_dict'} or node.attr in {'eval','query','ctypes','save','load','loadtxt','savetxt','fromfile','memmap'}):\n            raise ValueError('Forbidden attribute '+node.attr)\n    # Only the selected function is compiled. Permit pd/np imports by removing them from its body.\n    class RemoveImports(ast.NodeTransformer):\n        def visit_Import(self,node):return None\n    function=RemoveImports().visit(function)\n    tree=ast.fix_missing_locations(ast.Module(body=[function],type_ignores=[]))\n    return ast.unparse(tree)+'\\n'\n\n", 'sql_features.py': '"""Restricted SQL aggregates. The host owns tables, joins, time filtering and row order."""\nimport json\nimport re\nimport duckdb\nimport pandas as pd\n\nITEMS = [\'product_code\',\'product_type_no\',\'graphical_appearance_no\',\'colour_group_code\',\n         \'perceived_colour_value_id\',\'perceived_colour_master_id\',\'department_no\',\n         \'index_group_no\',\'section_no\',\'garment_group_no\']\nKEYS = {\'customer_id\',\'article_id\',*ITEMS}\nCOLUMNS = KEYS | {\'price\',\'days_ago\',\'sales_channel_id\',\'customer_age\'}\nFUNCTIONS = {\'count\',\'sum\',\'avg\',\'min\',\'max\',\'stddev_pop\',\'stddev_samp\',\'variance\',\'median\',\n             \'quantile_cont\',\'mode\',\'coalesce\',\'nullif\',\'abs\',\'sqrt\',\'ln\',\'log\',\'log2\',\'log10\',\n             \'exp\',\'pow\',\'power\',\'round\',\'floor\',\'ceil\',\'cast\',\'greatest\',\'least\',\'approx_count_distinct\'}\nWORDS = {\'case\',\'when\',\'then\',\'else\',\'end\',\'distinct\',\'filter\',\'where\',\'and\',\'or\',\n         \'not\',\'is\',\'null\',\'between\',\'in\',\'true\',\'false\',\'as\',\'double\',\'float\',\'integer\',\'bigint\'}\nTOKEN = re.compile(r\'\\s+|[A-Za-z][A-Za-z_0-9]*|\\d+(?:\\.\\d+)?|<=|>=|<>|!=|[(),+*/%=<>-]\')\n\n\nRAW = {\'raw_u_age\',\'raw_u_fn\',\'raw_u_active\',\'raw_u_club_member_status\',\'raw_u_fashion_news_frequency\',\n       *(\'raw_i_\'+name for name in ITEMS)}\n\n\ndef validate_expression(sql,columns=None,aggregate=True):\n    columns=COLUMNS if columns is None else columns\n    if not isinstance(sql,str) or not sql or len(sql)>1000:\n        raise ValueError(\'sql must be a nonempty numeric expression <=1000 characters\')\n    tokens=[]; offset=0\n    while offset<len(sql):\n        match=TOKEN.match(sql,offset)\n        if not match:\n            raise ValueError(\'Only numeric SQL expressions; no statements, strings, comments, or qualified names\')\n        offset=match.end()\n        if not match.group().isspace():\n            tokens.append(match.group().lower())\n    for i,token in enumerate(tokens):\n        if re.fullmatch(\'[a-z][a-z_0-9]*\',token):\n            if token not in columns|FUNCTIONS|WORDS:\n                raise ValueError(\'Unknown or forbidden SQL identifier: \'+token)\n            if i+1<len(tokens) and tokens[i+1]==\'(\' and token not in FUNCTIONS|{\'filter\',\'in\',\'and\',\'or\',\'not\',\'where\'}:\n                raise ValueError(\'Function outside allowlist: \'+token)\n    # IDs can be counted, never averaged or used as numerical predictions.\n    masked=re.sub(r\'\\b(?:count|approx_count_distinct)\\s*\\(\\s*(?:distinct\\s+)?(?:customer_id|article_id)\\s*\\)\',\n                  \'count(*)\',sql,flags=re.I) if aggregate else sql\n    if re.search(r\'\\b(customer_id|article_id)\\b\',masked,re.I):\n        raise ValueError(\'IDs only in grouping keys or count(DISTINCT id)\')\n    return \' \'.join(tokens)\n\n\ndef validate_program(program,allow_empty=False):\n    if not isinstance(program,dict) or set(program)-{\'rationale\',\'aggregates\',\'derived\'}:\n        raise ValueError(\'Return JSON with aggregates, derived and optional rationale only\')\n    entries=program.get(\'aggregates\',[]); derived=program.get(\'derived\',[])\n    if not isinstance(entries,list) or not isinstance(derived,list) or len(entries)>24 or len(derived)>12:\n        raise ValueError(\'Maximum24 aggregates and12 derived features, as in local Qwen interface\')\n    if not entries and not derived and not allow_empty:\n        raise ValueError(\'Provide at least one generated feature; raw control is host-only\')\n    names=set(); output=[]\n    for entry in entries:\n        if not isinstance(entry,dict) or set(entry)!={\'name\',\'keys\',\'sql\'}:\n            raise ValueError(\'Each aggregate needs exactly name, keys, sql\')\n        name=entry[\'name\']; keys=entry[\'keys\']\n        if not isinstance(name,str) or not re.fullmatch(\'f_[a-z][a-z0-9_]{0,45}\',name) or name in names:\n            raise ValueError(\'Use unique f_lowercase feature names\')\n        names.add(name)\n        if not isinstance(keys,list) or not 1<=len(keys)<=2 or any(not isinstance(k,str) or k not in KEYS for k in keys) or len(set(keys))!=len(keys):\n            raise ValueError(\'Invalid grouping keys\')\n        if len(keys)==2 and \'customer_id\' not in keys:\n            raise ValueError(\'Two keys must be customer_id + article_id or clothing category\')\n        output.append({\'name\':name,\'keys\':sorted(keys),\'sql\':validate_expression(entry[\'sql\'])})\n    formulas=[]; available=RAW|names\n    for entry in derived:\n        if not isinstance(entry,dict) or set(entry)!={\'name\',\'sql\'}:\n            raise ValueError(\'Each derived feature needs exactly name, sql\')\n        name=entry[\'name\']\n        if not isinstance(name,str) or not re.fullmatch(\'f_[a-z][a-z0-9_]{0,45}\',name) or name in names:\n            raise ValueError(\'Use unique f_lowercase feature names\')\n        formulas.append({\'name\':name,\'sql\':validate_expression(entry[\'sql\'],columns=available,aggregate=False)})\n        names.add(name);available.add(name)\n    return {\'aggregates\':sorted(output,key=lambda e:e[\'name\']),\'derived\':formulas}\n\n\ndef compile_sql(source):\n    program=validate_program(json.loads(source),allow_empty=True)\n\n    def modify_features(X,history):\n        result=pd.DataFrame(index=X.index)\n        if not program[\'aggregates\'] and not program[\'derived\']:\n            return result\n        connection=duckdb.connect(config={\'enable_external_access\':\'false\',\'threads\':\'1\',\n                                         \'memory_limit\':\'512MB\'})\n        try:\n            connection.register(\'history_input\',history)\n            groups={}\n            for entry in program[\'aggregates\']:\n                groups.setdefault(tuple(entry[\'keys\']),[]).append(entry)\n            for keys,entries in groups.items():\n                query=\'SELECT \'+\',\'.join(keys)+\',\'+\',\'.join(e[\'sql\']+\' AS \'+e[\'name\'] for e in entries)\n                query+=\' FROM history_input GROUP BY \'+\',\'.join(keys)\n                aggregate=connection.execute(query).df()\n                left=pd.DataFrame(index=X.index)\n                for key in keys:\n                    left[key]=X[key] if key in {\'customer_id\',\'article_id\'} else X[\'raw_i_\'+key]\n                joined=left.merge(aggregate,on=list(keys),how=\'left\',sort=False,validate=\'many_to_one\')\n                if len(joined)!=len(X):\n                    raise ValueError(\'Host SQL join changed row count\')\n                for entry in entries:\n                    result[entry[\'name\']]=joined[entry[\'name\']].to_numpy()\n            # Derived expressions see ONLY raw fields and previously computed features.\n            frame=pd.concat([X[[c for c in X if c.startswith(\'raw_\')]],result],axis=1)\n            for entry in program[\'derived\']:\n                connection.register(\'derived_input\',frame.reset_index(drop=True))\n                values=connection.execute(\'SELECT \'+entry[\'sql\']+\' AS \'+entry[\'name\']+\' FROM derived_input\').df()\n                connection.unregister(\'derived_input\')\n                if len(values)!=len(X):\n                    raise ValueError(\'Derived SQL must produce one scalar per candidate row\')\n                frame[entry[\'name\']]=values[entry[\'name\']].to_numpy()\n                result[entry[\'name\']]=values[entry[\'name\']].to_numpy()\n        finally:\n            connection.close()\n        return result[[e[\'name\'] for e in program[\'aggregates\']+program[\'derived\']]]\n    modify_features._feature_source=source\n    return modify_features\n', 'archive_helper.py': '"""Small frozen-prompt diagnostic, not an evolutionary reproduction."""\nimport ast\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedKFold\nfrom sklearn.preprocessing import LabelEncoder\n\n\ndef write_json(path,value):\n    Path(path).write_text(json.dumps(value,ensure_ascii=False,indent=2)+\'\\n\',encoding=\'utf-8\')\n\n\ndef pack_results(out,error=None,stage=None):\n    import shutil\n    import traceback\n    out=Path(out)\n    if error is not None:\n        write_json(out/\'failure.json\',{\'status\':\'failed\',\'stage\':stage,\'error\':str(error),\n            \'traceback\':\'\'.join(traceback.format_exception(type(error),error,error.__traceback__)),\n            \'llm_calls_started\':len(list(out.glob(\'*_seed*/request.json\')))})\n    return shutil.make_archive(str(out.parent/(out.name+\'_results\')),\'zip\',root_dir=out)\n\n\ndef initial_source(root):\n    spec=(Path(root)/\'specs/specification_balance-scale.txt\').read_text()\n    return spec.split(\'@equation.evolve\\n\',1)[1]\n\n\ndef build_prompt(root):\n    root=Path(root);spec=(root/\'specs/specification_balance-scale.txt\').read_text()\n    template=ast.literal_eval(ast.parse(spec).body[0].value)\n    frame=pd.read_csv(root/\'data/balance-scale.csv\')\n    y=LabelEncoder().fit_transform(frame.iloc[:,-1])\n    rows,_=next(StratifiedKFold(n_splits=5,shuffle=True,random_state=42).split(frame.iloc[:,:-1],y))\n    train=frame.iloc[rows].reset_index(drop=True)\n    descriptions=\'\\n\'.join(\'- \'+column+\': numerical variable within range [1, 5]\' for column in train.columns[:-1])\n    examples=[]\n    for i in train.sample(10,random_state=42).index:\n        parts=[column+\' is \'+str(train.loc[i,column]) for column in train.columns[:-1]]\n        examples.append(\'If \'+\', \'.join(parts)+\', Then Result is \'+str(y[rows[i]])+\'.\')\n    for key,value in {\'PREFIX\':(root/\'prompts/domain_head.txt\').read_text(),\n                      \'SUFFIX\':(root/\'prompts/domain_tail.txt\').read_text(),\n                      \'FEATURES\':descriptions,\'EXAMPLES\':\'\\n\'.join(examples)}.items():\n        template=template.replace(\'[\'+key+\']\',value)\n    seed=initial_source(root).replace(\'def modify_features(\', \'def modify_features_v0(\')\n    header=\'def modify_features_v1(df_input) -> pd.DataFrame:\\n    """Improved version of `modify_features_v0`. Think and suggest new features."""\\n\'\n    instruction=(\'You are a helpful assistant tasked with discovering new features/ dropping less important feaures \'\n                 \'for the given prediction task. Complete the modify_features function below, \'\n                 \'considering the physical meaning and relationships of inputs.\\n\\n\')\n    return instruction+template+\'\\n\'+seed+\'\\n\'+header\n\n\ndef evaluate(root,directory):\n    directory=Path(directory)\n    if (directory/\'result.json\').exists():return json.loads((directory/\'result.json\').read_text())\n    environment={k:v for k,v in os.environ.items() if not any(t in k.upper() for t in [\'TOKEN\',\'SECRET\',\'API_KEY\'])}\n    environment.update(CUDA_VISIBLE_DEVICES=\'\',OMP_NUM_THREADS=\'1\',OPENBLAS_NUM_THREADS=\'1\',MKL_NUM_THREADS=\'1\')\n    command=[sys.executable,str(Path(__file__).with_name(\'worker.py\')),\'--data\',str(Path(root)/\'data/balance-scale.csv\'),\'--directory\',str(directory)]\n    try:\n        with (directory/\'worker.log\').open(\'w\') as log:\n            completed=subprocess.run(command,env=environment,stdout=log,stderr=subprocess.STDOUT,timeout=90)\n        if completed.returncode or not (directory/\'result.json\').exists():\n            raise RuntimeError((directory/\'worker.log\').read_text()[-3000:])\n        return json.loads((directory/\'result.json\').read_text())\n    except Exception as error:\n        result={\'status\':\'rejected\',\'phase\':\'worker\',\'error\':str(error)}\n        write_json(directory/\'result.json\',result)\n        return result\n\n\ndef controls(root,out):\n    results={}\n    for name,source in {\'raw\':\'def modify_features(df_input):\\n    return df_input.copy()\\n\',\n                        \'authors_seed\':initial_source(root)}.items():\n        directory=Path(out)/name;directory.mkdir(exist_ok=True)\n        (directory/\'response.txt\').write_text(source)\n        result=evaluate(root,directory)\n        if result[\'status\']!=\'valid\':raise RuntimeError(result)\n        results[name]=result\n        print(name,\'train CV accuracy:\',result[\'cv_accuracy\'],flush=True)\n    write_json(Path(out)/\'controls.json\',results)\n    return results\n\n\ndef run_diagnostic(root,out,config,generate):\n    out=Path(out);prompt=build_prompt(root)\n    (out/\'prompt_base.txt\').write_text(prompt)\n    trials=[]\n    for mode in config[\'formats\']:\n        for seed in config[\'seeds\']:\n            directory=out/f\'{mode}_seed{seed}\';directory.mkdir(exist_ok=True)\n            (directory/\'prompt.txt\').write_text(prompt)\n            if (directory/\'trial.json\').exists():\n                trials.append(json.loads((directory/\'trial.json\').read_text()));continue\n            trial={\'format\':mode,\'seed\':seed,\'directory\':str(directory)}\n            try:\n                response_path=directory/\'response.json\'\n                if response_path.exists():response=json.loads(response_path.read_text())\n                else:\n                    if (directory/\'request.json\').exists():raise RuntimeError(\'Interrupted request consumes diagnostic slot; not silently regenerated\')\n                    write_json(directory/\'request.json\',trial)\n                    print(\'LLM call\',len(trials)+1,\'/6:\',mode,\'seed\',seed,flush=True)\n                    response=generate(prompt,seed,mode)\n                    write_json(response_path,response)\n                trial.update({k:v for k,v in response.items() if k not in {\'text\',\'rendered_prompt\'}})\n                (directory/\'response.txt\').write_text(response[\'text\'])\n                (directory/\'rendered_prompt.txt\').write_text(response.get(\'rendered_prompt\',prompt))\n                if response.get(\'truncated\'):raise ValueError(\'Truncated at output token limit\')\n                trial.update(evaluate(root,directory))\n            except Exception as error:trial.update(status=\'rejected\',error=str(error))\n            write_json(directory/\'trial.json\',trial)\n            trials.append(trial);write_json(out/\'trials.json\',trials)\n            print(mode,seed,trial[\'status\'],\'CV:\',trial.get(\'cv_accuracy\'),\'error:\',trial.get(\'error\',\'\'),flush=True)\n    return trials\n\n\ndef report(out,config,trials,control_results):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    out=Path(out);summary=[]\n    for mode in config[\'formats\']:\n        subset=[t for t in trials if t[\'format\']==mode]\n        summary.append({\'format\':mode,\'calls\':len(subset),\'valid\':sum(t[\'status\']==\'valid\' for t in subset),\n                        \'changed_features\':sum(t.get(\'has_feature_change\',False) for t in subset),\n                        \'truncated\':sum(t.get(\'truncated\',False) for t in subset)})\n    fig,axes=plt.subplots(1,2,figsize=(10,4))\n    axes[0].bar([s[\'format\'] for s in summary],[s[\'valid\'] for s in summary]);axes[0].set(ylabel=\'Executable /3 calls\',ylim=(0,3))\n    for mode in config[\'formats\']:\n        values=[t[\'cv_accuracy\'] for t in trials if t[\'format\']==mode and t[\'status\']==\'valid\']\n        if values:axes[1].scatter([mode]*len(values),values,label=mode)\n    for name,color in [(\'raw\',\'gray\'),(\'authors_seed\',\'green\')]:\n        axes[1].axhline(control_results[name][\'cv_accuracy\'],label=name,color=color)\n    axes[1].set_ylabel(\'Train-only4-fold CV accuracy\');axes[1].legend()\n    fig.suptitle(\'Balance-scale diagnostic — NOT paper reproduction / NOT H&M\');fig.tight_layout()\n    fig.savefig(out/\'diagnostic.png\',dpi=150);plt.close(fig)\n    write_json(out/\'summary.json\',{\'diagnostic_only\':True,\'paper_reproduction\':False,\'formats\':summary,\n        \'heldout_evaluated\':False,\'h_and_m_evaluated\':False,\'controls\':control_results})\n    lines=[\'# Balance-scale Llama diagnostic\',\'\',pd.DataFrame(summary).to_string(index=False),\'\',\n           \'No H&M or heldout result. Executable output alone does not establish useful feature discovery.\',\n           \'Chat/completion differences suggest format sensitivity;3 draws do not establish causality.\',\n           \'If both fail, inspect responses and model configuration before more H&M search.\']\n    (out/\'report.md\').write_text(\'\\n\'.join(lines))\n    pd.DataFrame([{k:t.get(k) for k in [\'format\',\'seed\',\'status\',\'cv_accuracy\',\'has_feature_change\',\'error\']} for t in trials]).to_csv(out/\'trials.csv\',index=False)\n    return summary\n'}
CODE = WORK / 'llmfe_flat_runtime'
CODE.mkdir(exist_ok=True)
for name, source in MODULES.items():
    (CODE / name).write_text(source, encoding='utf-8')
sys.path.insert(0, str(CODE))
import importlib
import worker as feature_worker
importlib.reload(feature_worker)
import runtime as experiment
importlib.reload(experiment)


In [ ]:
DATA, manifest = experiment.locate_bundle(INPUT_ROOT, WORK, '3d042b7fb24e702cb0784a99c6f04f82774e70dc802f710744232cddf016372a')
assert manifest['train_pairs'] == 803377 and manifest['train_user_times'] == 50000
assert manifest['files']['train_pairs.parquet']['sha256'] == manifest['train_source_sha256']
CONFIG['bundle_manifest_sha256'] = '3d042b7fb24e702cb0784a99c6f04f82774e70dc802f710744232cddf016372a'
display(pd.DataFrame([{'split':k,'users':v['users'],'pairs':v['pairs']} for k,v in manifest['splits'].items()]))
print('Training pairs:', manifest['train_pairs'], 'history rows:', manifest['history_rows'])
print('Data checksums: OK')

from flat_features import prepare
SOURCE_DATA=DATA
try:
    DATA,flat_manifest=prepare(SOURCE_DATA,WORK)
except Exception as error:
    from archive_helper import pack_results,write_json
    failed=WORK/('hm_flat_preparation_failed_'+str(time.time_ns()));failed.mkdir()
    write_json(failed/'config.json',CONFIG)
    display(FileLink(pack_results(failed,error,stage='table_preparation')))
    raise
CONFIG['flat_manifest_sha256']=hashlib.sha256((DATA/'flat_manifest.json').read_bytes()).hexdigest()
CONFIG['base_feature_origin']='15raw+15human historical aggregates'
print('Prepared flat fields:',flat_manifest['base_fields'])


## Доступ к Llama и фиксация запуска

`unsloth` скачивается анонимно, без чтения Kaggle Secrets и сохранённых HF-токенов.
Это полноразмерная публикация весов; NF4-квантизация выполняется при загрузке одинаково
для обоих источников. Unsloth-библиотека и её отдельное окружение не требуются.
`hf` требует одобрения Meta и `HF_TOKEN`. При403 проверьте доступ, при401 — токен.
Автоматического переключения источника нет. Ревизия публикации фиксируется отдельно;
побитовое совпадение весов двух издателей здесь не проверено. Лицензия Llama сохраняется.
Конфигурация, ревизия модели, версии пакетов, хеши данных и evaluator определяют новый run ID;
старые результаты не перезаписываются. Токен не сохраняется в артефакты и не выводится.


In [ ]:
import torch
from huggingface_hub import HfApi, hf_hub_download
if not torch.cuda.is_available():
    raise RuntimeError('Включите GPU T4 в настройках Kaggle')
gpu = torch.cuda.get_device_properties(0)
if gpu.total_memory < 14 * 1024**3:
    raise RuntimeError('Нужно не менее14GiB GPU RAM; выбранная карта не подходит для этой конфигурации')

def load_kaggle_secret():
    from kaggle_secrets import UserSecretsClient
    return UserSecretsClient().get_secret('HF_TOKEN')

token = resolve_auth(MODEL_SOURCE, os.environ, load_kaggle_secret)
try:
    revision = HfApi(token=token).model_info(MODEL_ID, revision=MODEL_REVISION or 'main').sha
    hf_hub_download(MODEL_ID, 'config.json', revision=revision, token=token)
except Exception:
    hint = 'Internet и существование ревизии' if MODEL_SOURCE == 'unsloth' else 'Internet, HF_TOKEN и одобрение доступа Meta'
    raise RuntimeError('Не удалось прочитать ' + MODEL_ID + '. Проверьте ' + hint) from None

CONFIG['model_revision'] = revision
CONFIG['gpu'] = gpu.name
CONFIG['versions'] = {p:importlib.metadata.version(p) for p in
    ['torch','transformers','accelerate','bitsandbytes','lightgbm','pandas','numpy','pyarrow','duckdb']}
CONFIG['runtime_sha256'] = {name:hashlib.sha256(source.encode()).hexdigest() for name,source in MODULES.items()}
fingerprint = hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:12]
RUN_ID = CONFIG['protocol'] + '_' + fingerprint
OUT = WORK / RUN_ID
OUT.mkdir(exist_ok=True)
config_path = OUT / 'run_config.json'
if config_path.exists() and json.loads(config_path.read_text()) != CONFIG:
    raise RuntimeError('Config mismatch: start a new run directory')
experiment.write_json(config_path, CONFIG)
print('Run:', RUN_ID, '| source:', MODEL_SOURCE, '| model:', MODEL_ID, '| GPU:', gpu.name, '| revision:', revision)


In [ ]:
try:
    # Prepared30 control, no LLM transformations.
    control = OUT / 'raw_control'
    control.mkdir(exist_ok=True)
    (control / 'program.py').write_text(experiment.RAW, encoding='utf-8')
    baseline = experiment.evaluate(DATA, control, 'search', CONFIG)
    print('Prepared30 LightGBM search MAP@12:', 100 * baseline['map'], '%')
    print('Control training/evaluation seconds:', baseline['seconds'])

    from flat_features import SEED, RAW_ONLY
    for name,code in [('seed_control',SEED),('raw15_control',RAW_ONLY)]:
        directory=OUT/name;directory.mkdir(exist_ok=True)
        (directory/'program.py').write_text(code)
        print(name,experiment.evaluate(DATA,directory,'search',CONFIG)['map'],flush=True)
except Exception as error:
    from archive_helper import pack_results
    display(FileLink(pack_results(OUT,error,stage='controls_model_or_final')))
    raise


## Загрузка Llama на GPU0

Это инференс, без fine-tuning. NF4 позволяет разместить веса и KV-кеш на одной T4.
Используется float16; BF16 и FlashAttention2 не требуются. CPU-деревья обучаются
в отдельном процессе и не занимают VRAM. Первое скачивание весов может быть долгим.


In [ ]:
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
    set_seed(CONFIG['seed'])
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=revision, token=token, trust_remote_code=False)
    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=revision, token=token,
        quantization_config=quantization, torch_dtype=torch.float16, device_map={'':0},
        attn_implementation='sdpa', trust_remote_code=False, low_cpu_mem_usage=True)
    model.eval()
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'
    print('Model GPU footprint GiB:', model.get_memory_footprint()/1024**3)
    experiment.write_json(OUT / 'model_runtime.json', {
        'model':MODEL_ID, 'model_source':MODEL_SOURCE, 'revision':revision, 'is_loaded_in_4bit':bool(model.is_loaded_in_4bit),
        'gpu':gpu.name, 'footprint_bytes':int(model.get_memory_footprint()),
        'allocated_vram_bytes':int(torch.cuda.memory_allocated(0))})
except Exception as error:
    from archive_helper import pack_results
    display(FileLink(pack_results(OUT,error,stage='controls_model_or_final')))
    raise


In [ ]:
def generate_program(prompt, seed):
    set_seed(seed)
    messages = [
        {'role':'system','content':'You are a feature engineering researcher. Follow the Python interface exactly.'},
        {'role':'user','content':prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt', add_special_tokens=False).to('cuda:0')
    prompt_tokens = inputs['input_ids'].shape[1]
    if prompt_tokens > CONFIG['max_input_tokens']:
        raise ValueError('Prompt exceeds input budget; no silent context truncation')
    endings = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')]
    endings = list({int(i) for i in endings if i is not None})
    tick = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=CONFIG['max_new_tokens'],
            do_sample=True, temperature=CONFIG['temperature'], top_p=CONFIG['top_p'], top_k=CONFIG['top_k'],
            num_return_sequences=1, eos_token_id=endings, pad_token_id=tokenizer.pad_token_id, use_cache=True)
    generated = output[0, prompt_tokens:]
    result = {'text':tokenizer.decode(generated, skip_special_tokens=True),
        'seed':seed, 'prompt_tokens':int(prompt_tokens), 'generated_tokens':int(len(generated)),
        'seconds':time.perf_counter()-tick,
        'truncated':len(generated)>=CONFIG['max_new_tokens'] and int(generated[-1]) not in endings}
    del output, inputs, generated
    return result

# b=3: одинаковый prompt, три независимых последовательных сэмпла; это экономит VRAM T4.
# Цель20 различных наборов с LightGBM MAP; максимум300 обращений, включая исправления.
# Одно исходное предложение + до3 исправлений = максимум4 обращения на цепочку.
# Ошибки/дубликаты не увеличивают число оценённых наборов. Raw control тоже не входит в20.
try:
    selected = experiment.search(DATA, OUT, CONFIG, generate_program)
except Exception as error:
    from archive_helper import pack_results
    display(FileLink(pack_results(OUT,error,stage='search')))
    raise
print('Selection frozen before test:', selected)
print('Search budget:', selected['search_budget'])
if not selected['search_budget']['target_met']:
    print('ВНИМАНИЕ: бюджет обращений исчерпан, цель20 наборов не достигнута. Итог будет по доступным наборам.')
display(Image(filename=str(OUT / 'search_progress.png')))
display(Image(filename=str(OUT / 'training_and_rejections.png')))


## Программа, выбранная LightGBM → итоговая оценка

Выбранные признаки фиксируются по confirmation LightGBM; test не влияет на выбор. Контроль без новых признаков
тоже может победить. Оба новых LightGBM обучаются на тех же803377 парах, что прежние
ручной и GNN→LightGBM. Все методы оцениваются на одинаковых val/test строках и пулах.
Старые модели не перетренируются: их сохранённые top12 включены в архив данных.
Протоколы подбора/производные seeds различаются, GNN обучалась дополнительно на графе.


In [ ]:
try:
    # Генерации больше не нужны; освобождаем VRAM.
    del model
    gc.collect()
    torch.cuda.empty_cache()
    report = experiment.final(DATA, OUT, CONFIG)
    display(pd.read_csv(OUT / 'comparison.csv').assign(
        val_map=lambda x:100*x.val_map, test_map=lambda x:100*x.test_map))
    print('Все новые модели: полный повтор предсказаний из сохранённых весов подтверждён.')
except Exception as error:
    from archive_helper import pack_results
    display(FileLink(pack_results(OUT,error,stage='controls_model_or_final')))
    raise


In [ ]:
for name in ['comparison.png','ap_histograms.png','paired_comparison.png','lightgbm_comparison.png']:
    display(Image(filename=str(OUT / name)))
display(Markdown((OUT / 'report.md').read_text()))


In [ ]:
import shutil
archive = shutil.make_archive(str(WORK / (RUN_ID + '_results')), 'zip', root_dir=OUT)
display(FileLink(archive))
print('Скачайте ZIP из Output. Веса Llama и HF_TOKEN в него не включены.')


## Что смотреть после запуска

- `report.md`, `comparison.csv`: MAP на одной и той же небольшой выборке, включая три способа + контроли.
- `comparison.png`, `ap_histograms.png`, `paired_comparison.png`, `lightgbm_comparison.png`: сравнение и распределения.
- `search_progress.png`, `training_and_rejections.png`: поиск, отказы и train logloss; loss не выбирает победителя.
- `candidates/candidate_*/program.py`, `response.json`, `trial.json`: полные программы, ответы и причины отказов.
- `selected.json`, `confirmation.json`: проверяемая история выбора только по validation.
- `model.json`/`model.txt`, `schema.json`, `*_metrics.json`, `*_predictions.npy`: новые деревья и независимый повтор.
- `run_config.json`, `model_runtime.json`: версия Llama, квантизация, пакеты, хеши кода/данных.

Проверки AST и отдельный CPU-процесс ограничивают интерфейс, время и память, но не являются
полноценной изоляцией произвольного враждебного Python-кода. Generated code не получает токен,
метки или будущую историю через функцию. Новые LLM-признаки не заимствуются из ручных/GNN-признаков.
